# 15 — Tuned pattern viewer

All thirteen chart patterns drawn at their **calibrated detector settings**, one legend toggle each.

Where the settings came from: a sweep of 108 detector configurations x 9 market windows x 13 patterns
on BTCUSDT 15m, 2022-2026 (`experiments/phase15_pattern_tuning.py`), re-run on the causal detector.

### Read this before you trade anything you see here

The settings were chosen to make the shapes **well-formed and frequent enough to study** — at least
ten detections per window in every one of the nine windows, at the strictest geometry that still
clears that floor. They were deliberately chosen **without looking at forward returns**.

That is not excessive caution, it is the finding. Re-running the identical sweep after fixing a
lookahead bug moved the best-scoring configuration for **13 of 13 patterns** — including patterns
whose ranking across all 108 settings correlated at 0.98-0.999. The score surface is flat and its
peak is noise, so any "best settings by score" table would be a table of noise.

| what the research says | |
|---|---|
| do these patterns predict **direction**? | **No.** Hit rate **0.48** — below a coin flip, and below a shuffled null at 0.50 |
| do they predict **anything**? | **Yes — size.** 18-42% more forward range than normal, against a null at 0.97 |
| so what are they for? | A **risk signal**: expect a bigger move, size accordingly. Not a direction call |

Full evidence in [`docs/WHAT_WORKS.md`](../docs/WHAT_WORKS.md).

### What each pattern draws

| drawn | meaning |
|---|---|
| thick line | the **fitted trendlines**, extended across the shape |
| dots | the **pivots** those lines are fitted through (3+ — two points always fit, so two means nothing) |
| amber line | the **boundary that had to break** for the pattern to confirm |
| dotted, coloured | the **measured-move target** |
| dotted, grey | the **protective level** |

Nothing is drawn early: a shape appears only from the bar it **confirmed**, using pivots that were
already confirmed by then. Shapes therefore show up later than you would spot them by eye — that is
what was actually knowable at the time.


## 1. Config — edit here

In [ ]:
# the kernel starts in notebooks/, so put the repo root on the path (same as the other notebooks)
import sys, os, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant

SYMBOL, TF = "BTCUSDT", "15m"          # the settings were calibrated on BTC 15m
START, END = "2026-07-01", "2026-08-24"
SOURCE     = "binance"                 # or "dukascopy" with SYMBOL="XAUUSD"

# None = all thirteen. Name a subset to declutter, e.g. ["rising_wedge", "head_shoulders"].
PATTERNS = None

# which are ON when the chart opens; the rest start hidden. "sr_zones" toggles the zones.
START_VISIBLE = ("sr_zones",)

SHOW_SR_ZONES = True
ZONE_LR, CLUSTER_PCT, MIN_TOUCHES = 10, 0.15, 2
EXTEND, HEIGHT = 12, 900

## 2. The tuned settings

`min_touches=3` and `line_tol_atr=0.6` are shared by every pattern — they come from a scan pooled
over all thirteen, and both landed where the prior argument says they should. `atr_mult` and
`min_span_bars` are what actually differ per pattern.

`atr_mult` sizes the **zigzag**, which is a property of a whole detection pass rather than of one
pattern, so patterns calibrated to different values cannot come from a single call.
`chart_pattern_shapes_tuned` groups them by `atr_mult` and runs one pass per group — three passes
here, not thirteen.

In [ ]:
import pandas as pd
from quant.indicators import tuned_specs, TUNED_COMMON

print("shared by every pattern:", TUNED_COMMON)
pd.DataFrame(tuned_specs()).T[["atr_mult", "min_span_bars"]].sort_values(
    ["atr_mult", "min_span_bars"], ascending=[False, True])

shared by every pattern: {'min_touches_upper': 3, 'min_touches_lower': 3, 'min_height_atr': 4.0, 'line_tol_atr': 0.6}


## 3. Data

In [ ]:
from quant.data import get_ohlcv

df = get_ohlcv(SYMBOL, TF, start=START, end=END, source=SOURCE, tz="UTC").reset_index(drop=True)
print(f"{len(df):,} bars   {df.t.min()}  ->  {df.t.max()}")

[07:24:38] INFO | quant.data | cache HIT  binance BTCUSDT 15m -> pushdown load


5,185 bars   2026-07-01 00:00:00+00:00  ->  2026-08-24 00:00:00+00:00


## 4. What was found

A count before plotting. A pattern showing 0 usually means the window is too short for its
`min_span_bars` rather than anything being wrong — the wedges and the symmetrical triangle need 200
bars each, and no shape may exceed 25% of the loaded chart.

In [ ]:
from quant.indicators import chart_pattern_shapes_tuned

shapes = chart_pattern_shapes_tuned(df, patterns=PATTERNS)
if len(shapes):
    print(shapes.pattern.value_counts().to_string())
    med = shapes.span_bars.median()
    print(f"\n{len(shapes)} shapes | median width {med:.0f} bars "
          f"({100*med/len(df):.1f}% of the chart), widest {shapes.span_bars.max():.0f}")
else:
    print("nothing found — widen START/END, or drop to a faster TF")

shapes[["t", "pattern", "direction", "span_bars", "touches_upper", "touches_lower",
        "entry", "stop", "target"]].tail(12)

triple_top                6
broadening_top            5
broadening_bottom         5
symmetrical_triangle      5
triple_bottom             4
ascending_triangle        3
double_bottom             3
inverse_head_shoulders    3
head_shoulders            2
descending_triangle       1
double_top                1
falling_wedge             1
rising_wedge              1

40 shapes | median width 152 bars (2.9% of the chart), widest 486


## 5. Chart — toggle each pattern from the legend

`tuned=True` is what applies the per-pattern settings. Without it `pattern_figure` uses one shared
configuration for everything, which is what notebook 14 does.

In [ ]:
from quant.viz import pattern_figure

fig = pattern_figure(
    df, patterns=PATTERNS, tuned=True, start_visible=START_VISIBLE,
    sr_zones=SHOW_SR_ZONES, zones_left=ZONE_LR, zones_right=ZONE_LR,
    cluster_pct=CLUSTER_PCT, min_touches=MIN_TOUCHES, extend=EXTEND,
    title=f"{SYMBOL} {TF}  {START} to {END}  —  tuned settings", height=HEIGHT)
fig

### Open it in a browser (bigger, and it survives the notebook being closed)

In [ ]:
from quant.viz import open_in_browser

open_in_browser(fig)

---
## 6. Zoom to one pattern

Set `PICK` to a row number from section 4 to focus on that occurrence. This is the view that shows
whether a shape is real — you should see the trendlines resting on their pivots and price closing
through the amber boundary.

In [ ]:
PICK = -1        # -1 = the most recent
PAD  = 80        # bars of context each side

if len(shapes):
    row = shapes.iloc[PICK]
    lo = max(int(row.start_i) - PAD, 0)
    hi = min(int(row.confirm_i) + PAD, len(df) - 1)
    sub = df.iloc[lo:hi + 1].reset_index(drop=True)
    zoom = pattern_figure(
        sub, patterns=[row.pattern], tuned=True, start_visible=(row.pattern,),
        sr_zones=SHOW_SR_ZONES, zones_left=ZONE_LR, zones_right=ZONE_LR,
        cluster_pct=CLUSTER_PCT, min_touches=MIN_TOUCHES, extend=EXTEND,
        title=f"{row.pattern}  confirmed {row.t}", height=700)
    display(zoom)
    print(f"span {row.span_bars} bars | touches {row.touches_upper}/{row.touches_lower} "
          f"| entry {row.entry:.1f} stop {row.stop:.1f} target {row.target:.1f}")

span 375 bars | touches 3/3 | entry 77499.1 stop 67718.3 target 89280.8


---
## 7. Override one pattern

`per_pattern` takes the same shape as `tuned_specs()`, so start from it and change what you want.
Loosening `min_span_bars` is usually what you need on a short window; lowering `min_touches` below 3
is not recommended — that is the setting that produced the unusable shapes, and the pooled scan now
ranks it worst.

In [ ]:
custom = tuned_specs()
custom["symmetrical_triangle"] = {**custom["symmetrical_triangle"], "min_span_bars": 80}

alt = pattern_figure(
    df, patterns=["symmetrical_triangle"], per_pattern=custom,
    start_visible=("symmetrical_triangle",), sr_zones=False, extend=EXTEND,
    title="symmetrical triangle — min_span_bars lowered to 80", height=700)
# alt

---
**One last framing.** Every trendline here is fitted through 3+ swing pivots by RANSAC with an
envelope constraint, so nothing pokes through the line it is supposed to bound. That is what makes
these correspond to what a person would draw.

But a well-drawn pattern is not a profitable one. On the nine-window test, `chart_short_rr8` — the
strongest pattern configuration this project found — was positive in **5 of 9** windows, exactly
matching a metronome that enters on a fixed schedule with no signal at all. Deleting the best 5
trades of ~45 takes every configuration to about **-1.0**.

Use these to see structure and to anticipate **volatility**, not direction.